# Análise de Densidade - Coral-sol

Autor:  Viviane da Rosa Sommer

Entrega: 20/09/2024

## Objetivo:
Notebook para avaliar a densidade do coral-sol, fazendo a inferência dos modelos de duto e coral-sol e analisando a intersecção das máscaras de resultado

## Importações Necessárias

In [ ]:
!pip install ultralytics
!pip install opencv-python-headless
!pip install torch

import glob
import torch
import cv2
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt
import os
import torchvision

## Declaração de Constantes e Modelos

In [ ]:
DUCT_CLASS_ID = 0
RESIZED_DIM = 640

easy_duct_model = YOLO('runs/segment/train-facil-500-v1/weights/best.pt')
medium_duct_model = YOLO('runs/segment/train-medio-500-v1/weights/best.pt')
hard_duct_model = YOLO('runs/segment/train-dificil-500-v1/weights/best.pt')
coral_model = YOLO('runs/segment/train-coral-seg-v1-128/weights/best.pt')

## Funções Necessárias

In [ ]:
def process_subimages(img, model, subimg_size=128):
    """
    Processes an image by dividing it into subimages, running the model on each subimage, 
    and combining the resulting masks.

    Parameters:
        img (np.array): The input image.
        model (YOLO): The YOLO model used for detection.
        subimg_size (int): The size of the subimages to process.

    Returns:
        torch.Tensor: The final mask generated from the subimages.
    """
    height, width = img.shape[:2]
    final_mask = torch.zeros((height, width), dtype=torch.uint8)

    for y in range(0, height, subimg_size):
        for x in range(0, width, subimg_size):
            subimg_height = min(subimg_size, height - y)
            subimg_width = min(subimg_size, width - x)
            subimg = img[y:y + subimg_height, x:x + subimg_width]
            
            results = model(subimg, verbose=False)

            for result in results:
                if result.masks is not None:
                    masks = result.masks.data
                    boxes = result.boxes.data
                    classes = boxes[:, 5]
                    indices = torch.where(classes == DUCT_CLASS_ID)
                    subimg_masks = masks[indices]
                    final_subimg_mask = torch.any(subimg_masks, dim=0).int() * 255

                    final_mask[y:y + subimg_height, x:x + subimg_width] = final_subimg_mask[:subimg_height, :subimg_width].cpu()

    return final_mask

def process_results(results, image):
    """
    Processes the results obtained from the model, returning the first valid result.

    Parameters:
        results (list): List of model detection results.
        image (np.array): The input image.

    Returns:
        result: The first valid result with masks, or None if no valid result is found.
    """
    for result in results:
        if result.masks is not None:
            return result
    return None

def smooth_subimage(mask, feathering_size=55):
    """
    Smooths the edges of a mask using Gaussian blur and converts it back to binary.

    Parameters:
        mask (np.array): The input binary mask.
        feathering_size (int): The size of the Gaussian kernel used for smoothing.

    Returns:
        np.array: The smoothed and binarized mask.
    """
    smoothed = cv2.GaussianBlur(mask, (feathering_size, feathering_size), 0)
    
    _, binary_mask = cv2.threshold(smoothed, 127, 255, cv2.THRESH_BINARY)
    return binary_mask

## Processamento das imagens, salvando os resultados

In [ ]:
final_directory = "Parte 3 - Resultados"
os.makedirs(final_directory, exist_ok=True)

results_list = []

for i, filename in enumerate(glob.glob("Parte 3 - Amarra & Riser/**/*.jpg")):
    img = cv2.imread(filename)
    if img is None:
        print(f"Failed to load image: {filename}")
        continue
    
    base_height, base_width = img.shape[:2]

    best_result = None
    for model in [easy_duct_model, medium_duct_model, hard_duct_model]:
        duct_results = model(img, verbose=False)
        best_result = process_results(duct_results, img)
        if best_result is not None:
            break

    if best_result is not None and best_result.masks is not None:
        masks = best_result.masks.data
        boxes = best_result.boxes.data
        classes = boxes[:, 5]
        scores = boxes[:, 4]  

        duct_indices = torch.where(classes == DUCT_CLASS_ID)[0]
        duct_boxes = boxes[duct_indices]
        duct_masks = masks[duct_indices]
        
        if len(duct_boxes) > 0:
            nms_indices = torchvision.ops.nms(duct_boxes[:, :4], scores[duct_indices], iou_threshold=0.2)
            duct_boxes = duct_boxes[nms_indices]
            duct_masks = duct_masks[nms_indices]

            final_duct_mask = torch.any(duct_masks, dim=0).int() * 255
        else:
            final_duct_mask = torch.zeros((RESIZED_DIM, RESIZED_DIM), dtype=torch.uint8)
    else:
        final_duct_mask = torch.zeros((RESIZED_DIM, RESIZED_DIM), dtype=torch.uint8)
        
    final_duct_mask_np = final_duct_mask.cpu().numpy()
    resized_final_duct_mask = cv2.resize(final_duct_mask_np, (base_width, base_height), interpolation=cv2.INTER_NEAREST)
    
    drawn_img = img.copy()
    combined_coral_mask = np.zeros((base_height, base_width), dtype=np.uint8)
    if best_result is not None and duct_boxes is not None:
        for box in duct_boxes:
            x1, y1, x2, y2 = map(int, box[:4])
            subimg = img[y1:y2, x1:x2]
            final_coral_mask = process_subimages(subimg, coral_model)
            resized_final_coral_mask = cv2.resize(final_coral_mask.cpu().numpy(), (x2 - x1, y2 - y1), interpolation=cv2.INTER_NEAREST)
            resized_final_coral_mask = smooth_subimage(resized_final_coral_mask) 
            combined_coral_mask[y1:y2, x1:x2] = np.maximum(combined_coral_mask[y1:y2, x1:x2], resized_final_coral_mask)
            cv2.rectangle(drawn_img, (x1, y1), (x2, y2), (0, 255, 0), 2)  


    intersection = np.logical_and(resized_final_duct_mask == 255, combined_coral_mask == 255).astype(np.uint8) * 255
    intersection_area = np.sum(intersection == 255)
    duct_area = np.sum(resized_final_duct_mask == 255)
    coral_area = np.sum(combined_coral_mask == 255)
    
    results_list.append([filename, duct_area, coral_area, intersection_area])
    save_as = filename.split("/")[-1].split(".")[0]

    plt.figure(figsize=(20, 10))  

    plt.subplot(1, 5, 1)
    plt.title('Duct Mask (Resized)')
    plt.imshow(resized_final_duct_mask, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 5, 2)
    plt.title('Coral Mask')
    plt.imshow(combined_coral_mask, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 5, 3)
    plt.title('Intersection')
    plt.imshow(intersection, cmap='gray')
    plt.axis('off')
    
    plt.subplot(1, 5, 4)
    plt.title('Original')
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis('off')

    plt.tight_layout()  
    plt.savefig(final_directory + f"/densidade_{save_as}.png")  
    plt.close()
    
    cv2.imwrite(final_directory + f"/retangulo_duto_{save_as}.png", drawn_img)  
    
    red_overlay = img.copy()
    red_overlay[resized_final_duct_mask != 0] = [0, 0, 255] 
    output_img = cv2.addWeighted(img, 1, red_overlay, 0.8, 0)
    cv2.imwrite(final_directory + f"/segmentacao_duto_{save_as}.png", output_img)
    
    red_overlay = img.copy()
    red_overlay[combined_coral_mask != 0] = [0, 0, 255]  
    output_img = cv2.addWeighted(img, 1, red_overlay, 0.8, 0)
    cv2.imwrite(final_directory + f"/segmentacao_coral_{save_as}.png", output_img)


## Conversão do formato do notebook: .ipynb para .html

In [ ]:
!jupyter nbconvert --to html 'density.ipynb'